# 🤖 04 - Advanced Model Building

In this notebook, we'll train several machine learning models to predict customer churn using our engineered dataset.
We'll focus on **Algorithm Arsenal** (Logistic Regression, Random Forest, XGBoost), **Rigorous Evaluation** (Precision, Recall, F1-Score, ROC-AUC), and **Hyperparameter Optimization**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install xgboost
    from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", palette="muted")
import warnings
warnings.filterwarnings('ignore')

## 1. Load Engineered Data

In [ ]:
df = pd.read_csv('../data/telco_churn_engineered.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Train-Test Split

In [ ]:
X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 3. Algorithm Arsenal
Let's define a helper function to evaluate our models.

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}\n")
    
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc_auc}

### Logistic Regression (Interpretable Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
res_lr = evaluate_model(lr, X_test, y_test, "Logistic Regression")

### Random Forest (Ensemble Method)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
res_rf = evaluate_model(rf, X_test, y_test, "Random Forest")

### XGBoost (Advanced Gradient Boosting)

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, learning_rate=0.05, max_depth=5)
xgb.fit(X_train, y_train)
res_xgb = evaluate_model(xgb, X_test, y_test, "XGBoost")

## 4. Rigorous Evaluation & Comparison

In [ ]:
results = pd.DataFrame([res_lr, res_rf, res_xgb]).set_index('Model')
display(results)

results.plot(kind='bar', figsize=(12, 6), colormap='viridis')
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()